# 03 - Integración del modelo final en scripts y app

Este notebook sirve para **explicar y generar** los archivos `.py` finales del proyecto ConcertDemandAI.

> Importante: el proyecto debe conservar archivos `.py` para funcionar como aplicación y como scripts.  
> Este notebook es una versión explicativa para revisar el código y, si se ejecuta en Colab, escribir los archivos en el repositorio.


## 0. Ubicarse en la raíz del repositorio

Antes de generar los archivos `.py`, este notebook debe ejecutarse desde la raíz del proyecto.

Esto evita que Colab cree carpetas en `/content/model` o `/content/app` por error.  
La ruta correcta debe ser:

```text
/content/concert-demand-ml
```


In [ ]:
from pathlib import Path

repo_path = Path("/content/concert-demand-ml")

if repo_path.exists():
    %cd /content/concert-demand-ml
else:
    %cd /content
    !git clone https://github.com/leonciochiunti/concert-demand-ml.git
    %cd /content/concert-demand-ml

print("Ruta actual:")
!pwd

print("\nContenido del repositorio:")
!ls -lah


## Diferencia entre `.ipynb` y `.py`

Un archivo `.ipynb` es un **Jupyter Notebook**. Sirve para documentar, explicar, probar celdas por partes y mostrar resultados visuales. Es ideal para el análisis exploratorio y el entrenamiento experimental.

Un archivo `.py` es un **script de Python**. Sirve para que el proyecto funcione como software: entrenar desde terminal, cargar el modelo, hacer predicciones y ejecutar la app de Streamlit.

En este proyecto:

- `01_eda_training.ipynb` → análisis exploratorio.
- `02_train_model.ipynb` → entrenamiento explicado y comparación de modelos.
- `model/train.py` → script para volver a entrenar el modelo.
- `model/predict.py` → funciones para usar el modelo entrenado.
- `app/app.py` → aplicación Streamlit.
- `model/model.py` → constantes y feature engineering compartido.


## 1. Preparar carpetas del proyecto

Esta celda crea las carpetas `model/` y `app/` en caso de que no existan.

Se usa porque los archivos modificados deben quedar en esa estructura dentro del repositorio.


In [ ]:
from pathlib import Path

Path("model").mkdir(parents=True, exist_ok=True)
Path("app").mkdir(parents=True, exist_ok=True)

print("Carpetas listas: model/ y app/")


## 2. Crear `model/__init__.py`

Este archivo permite que la carpeta `model/` funcione como paquete de Python.

Es útil para poder importar funciones de esta forma:

```python
from model.predict import predict_event
```


In [ ]:
%%writefile model/__init__.py
"""
Paquete de utilidades del modelo de ConcertDemandAI.

Este archivo permite importar funciones desde la carpeta model/ como paquete:
from model.predict import predict_event
"""


## 3. Crear `model/model.py`

Este archivo centraliza:

- columnas usadas por el modelo;
- variables categóricas y numéricas;
- mapas simulados de artista, ciudad y país;
- normalización de días;
- cálculo de variables derivadas;
- preparación de datos para entrenamiento y predicción.

Se hizo así para que `train.py`, `predict.py` y `app.py` usen la misma lógica y no haya inconsistencias.


In [ ]:
%%writefile model/model.py
"""
model.py

Funciones y constantes compartidas para ConcertDemandAI.

¿Por qué existe este archivo?
- Para que train.py, predict.py y app.py usen las mismas columnas.
- Para evitar que cada archivo calcule las variables de forma diferente.
- Para mantener en un solo lugar los mapas de artista, ciudad, país y variables derivadas.

Nota importante:
Las variables históricas usadas aquí son simuladas para el MVP académico.
No representan afirmaciones reales sobre artistas o mercados.
"""

from __future__ import annotations

from typing import Any, Dict, Iterable, List

import pandas as pd


# ==============================
# 1. Columnas del modelo
# ==============================
# Estas columnas son las mismas que se usaron en el notebook 02_train_model.
# Se excluyen occupancy_pct y tickets_sold porque son variables posteriores al evento.
# Usarlas como entrada sería fuga de información.
FEATURES: List[str] = [
    "artist",
    "genre",
    "city",
    "country",
    "venue_type",
    "capacity",
    "month",
    "event_day",
    "days_until_event",
    "marketing_budget",
    "ticket_price",
    "artist_popularity",
    "previous_attendance_rate",
    "pre_sale_interest",
    "artist_market_score",
    "is_weekend",
    "marketing_per_capacity",
    "price_per_capacity",
    "popularity_marketing",
]

TARGET = "demand_class"

CATEGORICAL_FEATURES: List[str] = [
    "artist",
    "genre",
    "city",
    "country",
    "venue_type",
    "event_day",
]

NUMERIC_FEATURES: List[str] = [
    "capacity",
    "month",
    "days_until_event",
    "marketing_budget",
    "ticket_price",
    "artist_popularity",
    "previous_attendance_rate",
    "pre_sale_interest",
    "artist_market_score",
    "is_weekend",
    "marketing_per_capacity",
    "price_per_capacity",
    "popularity_marketing",
]


# ==============================
# 2. Mapas usados para enriquecer datos
# ==============================
# Estos valores se basan en el dataset simulado del proyecto.
# Sirven para estimar señales previas al evento cuando el usuario no las captura manualmente.
ARTIST_ATTENDANCE_RATE: Dict[str, float] = {
    "BTS": 0.95,
    "The Weeknd": 0.90,
    "Ariana Grande": 0.88,
    "Drake": 0.86,
    "TWICE": 0.84,
    "David Guetta": 0.82,
    "Grupo Firme": 0.78,
    "Los Tigres del Norte": 0.74,
    "Peso Pluma": 0.58,
}

CITY_MARKET_SCORE: Dict[str, float] = {
    "CDMX": 1.00,
    "Guadalajara": 0.92,
    "Monterrey": 0.92,
    "Queretaro": 0.82,
    "Querétaro": 0.82,
    "Leon": 0.78,
    "León": 0.78,
    "Puebla": 0.76,
    "Toluca": 0.72,
    "Los Angeles": 0.95,
    "Bogota": 0.88,
    "Bogotá": 0.88,
    "Madrid": 0.90,
}

CITY_COUNTRY: Dict[str, str] = {
    "CDMX": "Mexico",
    "Guadalajara": "Mexico",
    "Monterrey": "Mexico",
    "Queretaro": "Mexico",
    "Querétaro": "Mexico",
    "Leon": "Mexico",
    "León": "Mexico",
    "Puebla": "Mexico",
    "Toluca": "Mexico",
    "Los Angeles": "Estados Unidos",
    "Bogota": "Colombia",
    "Bogotá": "Colombia",
    "Madrid": "España",
}

DAY_NORMALIZATION: Dict[str, str] = {
    "monday": "Monday",
    "lunes": "Monday",
    "tuesday": "Tuesday",
    "martes": "Tuesday",
    "wednesday": "Wednesday",
    "miercoles": "Wednesday",
    "miércoles": "Wednesday",
    "thursday": "Thursday",
    "jueves": "Thursday",
    "friday": "Friday",
    "viernes": "Friday",
    "saturday": "Saturday",
    "sabado": "Saturday",
    "sábado": "Saturday",
    "sunday": "Sunday",
    "domingo": "Sunday",
}

WEEKEND_DAYS = {"Friday", "Saturday", "Sunday"}


# ==============================
# 3. Funciones de normalización
# ==============================
# Estas funciones hacen que la app pueda usar español en la interfaz,
# pero el modelo reciba valores compatibles con el entrenamiento.
def normalize_event_day(event_day: Any) -> str:
    """Convierte días en español o inglés a la forma usada por el modelo."""
    if event_day is None:
        return "Saturday"

    value = str(event_day).strip()
    return DAY_NORMALIZATION.get(value.lower(), value)


def infer_country(city: Any, country: Any = None) -> str:
    """Obtiene el país a partir de la ciudad si el usuario no lo proporciona."""
    if country is not None and str(country).strip():
        return str(country).strip()

    city_value = str(city).strip()
    return CITY_COUNTRY.get(city_value, "Mexico")


def infer_previous_attendance_rate(artist: Any, value: Any = None) -> float:
    """Obtiene el historial previo del artista si no viene en la entrada."""
    if value is not None:
        try:
            return float(value)
        except ValueError:
            pass

    artist_value = str(artist).strip()
    return float(ARTIST_ATTENDANCE_RATE.get(artist_value, 0.70))


def infer_artist_market_score(city: Any, value: Any = None) -> float:
    """Obtiene el score de mercado de la ciudad si no viene en la entrada."""
    if value is not None:
        try:
            return float(value)
        except ValueError:
            pass

    city_value = str(city).strip()
    return float(CITY_MARKET_SCORE.get(city_value, 0.80))


def estimate_pre_sale_interest(
    capacity: float,
    previous_attendance_rate: float,
    artist_market_score: float,
    artist_popularity: float,
    value: Any = None,
) -> float:
    """
    Estima interés de preventa cuando no se captura manualmente.

    ¿Por qué se calcula?
    El modelo final usa pre_sale_interest porque representa una señal previa al evento.
    En producción podría venir de búsquedas, registros, preventas o visitas.
    En el MVP académico se estima de manera determinista para que la app funcione.
    """
    if value is not None:
        try:
            return float(value)
        except ValueError:
            pass

    estimated = (
        capacity
        * previous_attendance_rate
        * artist_market_score
        * (artist_popularity / 100)
        * 0.85
    )

    # No permitimos valores negativos ni mayores a la capacidad del recinto.
    return float(max(0, min(capacity, round(estimated))))


# ==============================
# 4. Feature engineering
# ==============================
# Aquí se calculan variables derivadas que el modelo aprendió durante el entrenamiento.
def build_feature_row(event: Dict[str, Any]) -> Dict[str, Any]:
    """
    Construye una fila completa con todas las variables que espera el modelo.

    Entrada esperada mínima:
    artist, genre, city, venue_type, capacity, month, event_day,
    days_until_event, marketing_budget, ticket_price, artist_popularity.

    Salida:
    Diccionario con las 19 columnas de FEATURES.
    """
    capacity = float(event.get("capacity", 0))
    marketing_budget = float(event.get("marketing_budget", 0))
    ticket_price = float(event.get("ticket_price", 0))
    artist_popularity = float(event.get("artist_popularity", 0))

    artist = event.get("artist", "Unknown")
    genre = event.get("genre", "Unknown")
    city = event.get("city", "CDMX")
    venue_type = event.get("venue_type", "arena")
    event_day = normalize_event_day(event.get("event_day", "Saturday"))

    country = infer_country(city, event.get("country"))
    previous_attendance_rate = infer_previous_attendance_rate(
        artist,
        event.get("previous_attendance_rate"),
    )
    artist_market_score = infer_artist_market_score(
        city,
        event.get("artist_market_score"),
    )
    pre_sale_interest = estimate_pre_sale_interest(
        capacity=capacity,
        previous_attendance_rate=previous_attendance_rate,
        artist_market_score=artist_market_score,
        artist_popularity=artist_popularity,
        value=event.get("pre_sale_interest"),
    )

    is_weekend = 1 if event_day in WEEKEND_DAYS else 0

    # Evitamos división entre cero por seguridad.
    marketing_per_capacity = marketing_budget / capacity if capacity else 0
    price_per_capacity = ticket_price / capacity if capacity else 0
    popularity_marketing = artist_popularity * marketing_budget

    row = {
        "artist": artist,
        "genre": genre,
        "city": city,
        "country": country,
        "venue_type": venue_type,
        "capacity": capacity,
        "month": int(event.get("month", 1)),
        "event_day": event_day,
        "days_until_event": int(event.get("days_until_event", 0)),
        "marketing_budget": marketing_budget,
        "ticket_price": ticket_price,
        "artist_popularity": artist_popularity,
        "previous_attendance_rate": previous_attendance_rate,
        "pre_sale_interest": pre_sale_interest,
        "artist_market_score": artist_market_score,
        "is_weekend": is_weekend,
        "marketing_per_capacity": marketing_per_capacity,
        "price_per_capacity": price_per_capacity,
        "popularity_marketing": popularity_marketing,
    }

    return row


def build_feature_dataframe(events: Dict[str, Any] | Iterable[Dict[str, Any]]) -> pd.DataFrame:
    """Convierte uno o varios eventos a DataFrame con el orden correcto de columnas."""
    if isinstance(events, dict):
        rows = [build_feature_row(events)]
    else:
        rows = [build_feature_row(event) for event in events]

    df = pd.DataFrame(rows)

    # Reordenamos las columnas para que coincidan exactamente con el entrenamiento.
    return df[FEATURES]


def prepare_training_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara el dataset para entrenamiento.

    ¿Por qué existe?
    Si el dataset ya trae las variables históricas, las respeta.
    Si faltan algunas variables derivadas, las calcula para evitar errores.
    """
    prepared_rows = []

    for _, row in df.iterrows():
        event = row.to_dict()
        feature_row = build_feature_row(event)

        # Conservamos el target si existe.
        if TARGET in row:
            feature_row[TARGET] = row[TARGET]

        prepared_rows.append(feature_row)

    return pd.DataFrame(prepared_rows)


def validate_training_columns(df: pd.DataFrame) -> None:
    """Valida que el dataset tenga el target y las columnas necesarias."""
    if TARGET not in df.columns:
        raise ValueError(f"El dataset debe contener la columna target: {TARGET}")

    missing = [column for column in FEATURES if column not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas para entrenar: {missing}")


## 4. Crear `model/train.py`

Este script permite volver a entrenar el modelo final desde terminal:

```bash
python model/train.py
```

Se basa en el resultado del notebook `02_train_model`, donde el mejor modelo fue `Logistic Regression`.

El script guarda:

- `model/model.pkl`
- `model/metrics.txt`
- `model/metrics.json`


In [ ]:
%%writefile model/train.py
"""
train.py

Script de entrenamiento final para ConcertDemandAI.

¿Por qué este archivo?
- El notebook 02_train_model.ipynb sirve para experimentar y explicar.
- Este script sirve para repetir el entrenamiento desde terminal:
  python model/train.py

Modelo final:
- Logistic Regression
- Seleccionado porque fue el mejor modelo en F1 macro y accuracy en el notebook.

Métricas finales esperadas del notebook:
- Accuracy: 0.8500
- F1 macro: 0.8512
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, Any

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from .model import (
        CATEGORICAL_FEATURES,
        FEATURES,
        NUMERIC_FEATURES,
        TARGET,
        prepare_training_dataframe,
        validate_training_columns,
    )
except ImportError:
    from model import (
        CATEGORICAL_FEATURES,
        FEATURES,
        NUMERIC_FEATURES,
        TARGET,
        prepare_training_dataframe,
        validate_training_columns,
    )


# ==============================
# 1. Rutas del proyecto
# ==============================
# Se calculan de forma relativa para que funcione desde Colab, VS Code o terminal.
PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_PATH = PROJECT_ROOT / "data" / "dataset.csv"
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_PATH = MODEL_DIR / "model.pkl"
METRICS_TXT_PATH = MODEL_DIR / "metrics.txt"
METRICS_JSON_PATH = MODEL_DIR / "metrics.json"

RANDOM_STATE = 42


# ==============================
# 2. Carga y preparación del dataset
# ==============================
def load_dataset(path: Path = DATA_PATH) -> pd.DataFrame:
    """Carga el dataset y prepara las variables usadas por el modelo."""
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el dataset en: {path}")

    df = pd.read_csv(path)

    # Preparamos las variables históricas y derivadas.
    # Esto evita diferencias entre notebook, script y app.
    df = prepare_training_dataframe(df)

    validate_training_columns(df)

    return df


# ==============================
# 3. Construcción del pipeline
# ==============================
def build_pipeline() -> Pipeline:
    """
    Construye el pipeline final.

    ¿Por qué Pipeline?
    Porque guarda juntos:
    - preprocesamiento de variables categóricas y numéricas
    - modelo final Logistic Regression

    Así model.pkl puede transformar datos nuevos y predecir sin pasos manuales.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ("num", StandardScaler(), NUMERIC_FEATURES),
        ]
    )

    model = LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return pipeline


# ==============================
# 4. Evaluación del modelo
# ==============================
def evaluate_model(pipeline: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> Dict[str, Any]:
    """
    Calcula métricas del modelo final.

    ¿Por qué F1 macro?
    Porque el problema tiene tres clases: baja, media y alta.
    F1 macro evalúa el desempeño promedio entre clases sin favorecer una clase específica.
    """
    y_pred = pipeline.predict(X_test)

    metrics = {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "classification_report": classification_report(y_test, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "classes": list(pipeline.classes_),
        "features": FEATURES,
        "excluded_columns": [
            "occupancy_pct",
            "tickets_sold",
        ],
        "note": (
            "occupancy_pct y tickets_sold no se usan como variables de entrada "
            "porque son información posterior al evento y causarían fuga de información."
        ),
    }

    return metrics


# ==============================
# 5. Guardado de modelo y métricas
# ==============================
def save_artifacts(pipeline: Pipeline, metrics: Dict[str, Any]) -> None:
    """Guarda model.pkl, metrics.txt y metrics.json en la carpeta model/."""
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    joblib.dump(pipeline, MODEL_PATH)

    metrics_text = f"""Modelo final: {metrics['model']}

Accuracy: {metrics['accuracy']:.4f}
Precision macro: {metrics['precision_macro']:.4f}
Recall macro: {metrics['recall_macro']:.4f}
F1 macro: {metrics['f1_macro']:.4f}
F1 weighted: {metrics['f1_weighted']:.4f}

Reporte de clasificación:
{metrics['classification_report']}

Matriz de confusión:
{metrics['confusion_matrix']}

Nota:
{metrics['note']}
"""

    METRICS_TXT_PATH.write_text(metrics_text, encoding="utf-8")

    with METRICS_JSON_PATH.open("w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=4, ensure_ascii=False)


# ==============================
# 6. Ejecución principal
# ==============================
def main() -> None:
    """Entrena el modelo final y guarda los artefactos."""
    print("==============================")
    print("Entrenamiento del modelo final")
    print("==============================")

    df = load_dataset()

    X = df[FEATURES]
    y = df[TARGET]

    # Stratify mantiene la proporción de baja/media/alta en train y test.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(pipeline, X_test, y_test)
    save_artifacts(pipeline, metrics)

    print("Modelo entrenado y guardado correctamente.")
    print(f"Modelo: {MODEL_PATH}")
    print(f"Métricas TXT: {METRICS_TXT_PATH}")
    print(f"Métricas JSON: {METRICS_JSON_PATH}")
    print()
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"F1 macro: {metrics['f1_macro']:.4f}")


if __name__ == "__main__":
    main()


## 5. Crear `model/predict.py`

Este archivo carga `model.pkl` y permite hacer predicciones nuevas.

La app no debería tener que preparar manualmente todas las variables. Por eso `predict.py` recibe los datos principales del evento, llama a `model.py`, construye las variables necesarias y devuelve:

- predicción: `alta`, `media` o `baja`;
- probabilidades por clase;
- variables enviadas al modelo.


In [ ]:
%%writefile model/predict.py
"""
predict.py

Funciones de predicción para ConcertDemandAI.

¿Por qué este archivo?
- Permite usar model.pkl sin volver a entrenar.
- Centraliza la forma correcta de construir las variables antes de predecir.
- Sirve como puente entre el modelo entrenado y app.py.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Iterable

import joblib
import pandas as pd

try:
    from .model import build_feature_dataframe
except ImportError:
    from model import build_feature_dataframe


# ==============================
# 1. Ruta del modelo entrenado
# ==============================
PROJECT_ROOT = Path(__file__).resolve().parents[1]
MODEL_PATH = PROJECT_ROOT / "model" / "model.pkl"


# ==============================
# 2. Carga del modelo
# ==============================
def load_model(model_path: Path = MODEL_PATH):
    """Carga el modelo entrenado desde model.pkl."""
    if not model_path.exists():
        raise FileNotFoundError(
            f"No se encontró el modelo en {model_path}. "
            "Primero ejecuta python model/train.py o corre el notebook 02_train_model."
        )

    return joblib.load(model_path)


# ==============================
# 3. Predicción individual
# ==============================
def predict_event(event: Dict[str, Any], model=None) -> Dict[str, Any]:
    """
    Predice la demanda de un concierto.

    event puede traer solo las variables principales.
    Las variables derivadas se calculan en build_feature_dataframe.
    """
    if model is None:
        model = load_model()

    X = build_feature_dataframe(event)
    prediction = model.predict(X)[0]

    result: Dict[str, Any] = {
        "prediction": prediction,
        "input_features": X.iloc[0].to_dict(),
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)[0]
        result["probabilities"] = {
            class_name: float(probability)
            for class_name, probability in zip(model.classes_, probabilities)
        }

    return result


# ==============================
# 4. Predicción por lote
# ==============================
def predict_events(events: Iterable[Dict[str, Any]], model=None) -> pd.DataFrame:
    """Predice varios eventos y regresa un DataFrame con resultados."""
    if model is None:
        model = load_model()

    X = build_feature_dataframe(events)
    predictions = model.predict(X)

    results = X.copy()
    results["prediction"] = predictions

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        for index, class_name in enumerate(model.classes_):
            results[f"prob_{class_name}"] = probabilities[:, index]

    return results


# ==============================
# 5. Ejemplo rápido desde terminal
# ==============================
if __name__ == "__main__":
    example_event = {
        "artist": "The Weeknd",
        "genre": "pop",
        "city": "Guadalajara",
        "venue_type": "arena",
        "capacity": 18000,
        "month": 8,
        "event_day": "Friday",
        "days_until_event": 90,
        "marketing_budget": 180000,
        "ticket_price": 1800,
        "artist_popularity": 88,
    }

    output = predict_event(example_event)

    print("Predicción:", output["prediction"])
    print("Probabilidades:")
    for class_name, probability in output.get("probabilities", {}).items():
        print(f"{class_name}: {probability:.4f}")


## 6. Crear `app/app.py`

Este archivo contiene la app de Streamlit.

La app permite capturar datos de un concierto, usar el modelo entrenado y mostrar:

- predicción de demanda;
- probabilidades por clase;
- recomendación básica;
- métricas del modelo final;
- variables enviadas al modelo.

Este archivo debe mantenerse como `.py`, porque Streamlit se ejecuta con:

```bash
streamlit run app/app.py
```


In [ ]:
%%writefile app/app.py
"""
app.py

Interfaz Streamlit para ConcertDemandAI.

¿Por qué se actualizó?
- El modelo final ya no usa solamente variables básicas.
- Ahora necesita variables históricas y derivadas.
- La app debe capturar las variables principales y calcular el resto antes de predecir.
"""

from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import streamlit as st


# ==============================
# 1. Configuración de rutas
# ==============================
# Streamlit ejecuta app/app.py desde la carpeta app.
# Agregamos la raíz del proyecto para poder importar model.predict.
PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from model.predict import load_model, predict_event  # noqa: E402
from model.model import CITY_COUNTRY  # noqa: E402


MODEL_PATH = PROJECT_ROOT / "model" / "model.pkl"
METRICS_JSON_PATH = PROJECT_ROOT / "model" / "metrics.json"


# ==============================
# 2. Configuración visual
# ==============================
st.set_page_config(
    page_title="ConcertDemandAI",
    page_icon="🎵",
    layout="wide",
)

st.title("🎵 ConcertDemandAI")
st.caption("Predicción de demanda de conciertos usando Machine Learning")


# ==============================
# 3. Carga del modelo
# ==============================
# Se usa cache para no cargar model.pkl en cada interacción.
@st.cache_resource
def get_model():
    return load_model(MODEL_PATH)


try:
    model = get_model()
except Exception as error:
    st.error("No fue posible cargar el modelo entrenado.")
    st.info("Verifica que exista el archivo model/model.pkl.")
    st.exception(error)
    st.stop()


# ==============================
# 4. Carga de métricas
# ==============================
# Las métricas sirven para mostrar evidencia del desempeño del modelo final.
def load_metrics():
    if METRICS_JSON_PATH.exists():
        with METRICS_JSON_PATH.open("r", encoding="utf-8") as file:
            return json.load(file)
    return None


metrics = load_metrics()

if metrics:
    col1, col2, col3 = st.columns(3)
    col1.metric("Accuracy", f"{metrics.get('accuracy', 0):.4f}")
    col2.metric("F1 macro", f"{metrics.get('f1_macro', 0):.4f}")
    col3.metric("Modelo", metrics.get("model", "Logistic Regression"))


st.divider()


# ==============================
# 5. Formulario de entrada
# ==============================
# El usuario captura variables disponibles antes del evento.
# Las variables históricas y derivadas se calculan automáticamente en model.py.
st.subheader("Datos del concierto")

artist_options = [
    "BTS",
    "The Weeknd",
    "Ariana Grande",
    "Drake",
    "TWICE",
    "David Guetta",
    "Grupo Firme",
    "Los Tigres del Norte",
    "Peso Pluma",
]

genre_options = [
    "kpop",
    "pop",
    "rap",
    "edm",
    "regional_mexicano",
    "rock",
    "latin",
]

city_options = [
    "CDMX",
    "Guadalajara",
    "Monterrey",
    "Queretaro",
    "Leon",
    "Puebla",
    "Toluca",
    "Los Angeles",
    "Bogota",
    "Madrid",
]

venue_options = [
    "stadium",
    "arena",
    "theater",
    "club",
]

day_options = {
    "Lunes": "Monday",
    "Martes": "Tuesday",
    "Miércoles": "Wednesday",
    "Jueves": "Thursday",
    "Viernes": "Friday",
    "Sábado": "Saturday",
    "Domingo": "Sunday",
}


with st.form("prediction_form"):
    left, right = st.columns(2)

    with left:
        artist = st.selectbox("Artista", artist_options)
        genre = st.selectbox("Género", genre_options)
        city = st.selectbox("Ciudad", city_options)
        country = CITY_COUNTRY.get(city, "Mexico")
        st.text_input("País", value=country, disabled=True)
        venue_type = st.selectbox("Tipo de recinto", venue_options)
        capacity = st.number_input("Capacidad del recinto", min_value=1000, max_value=100000, value=18000, step=1000)

    with right:
        month = st.slider("Mes del evento", min_value=1, max_value=12, value=8)
        event_day_label = st.selectbox("Día del evento", list(day_options.keys()), index=5)
        days_until_event = st.number_input("Días para el evento", min_value=1, max_value=365, value=90, step=1)
        marketing_budget = st.number_input("Presupuesto de marketing", min_value=0, max_value=2_000_000, value=180000, step=10000)
        ticket_price = st.number_input("Precio del boleto", min_value=100, max_value=10000, value=1800, step=100)
        artist_popularity = st.slider("Popularidad del artista", min_value=0, max_value=100, value=88)

    submitted = st.form_submit_button("Predecir demanda")


# ==============================
# 6. Predicción
# ==============================
if submitted:
    event_data = {
        "artist": artist,
        "genre": genre,
        "city": city,
        "country": country,
        "venue_type": venue_type,
        "capacity": capacity,
        "month": month,
        "event_day": day_options[event_day_label],
        "days_until_event": days_until_event,
        "marketing_budget": marketing_budget,
        "ticket_price": ticket_price,
        "artist_popularity": artist_popularity,
    }

    result = predict_event(event_data, model=model)

    prediction = result["prediction"]
    probabilities = result.get("probabilities", {})

    st.divider()
    st.subheader("Resultado de predicción")

    if prediction == "alta":
        st.success("Demanda estimada: ALTA")
        recommendation = (
            "Recomendación: reforzar logística, inventario de boletos, campañas digitales "
            "y operación del recinto."
        )
    elif prediction == "media":
        st.warning("Demanda estimada: MEDIA")
        recommendation = (
            "Recomendación: monitorear preventa, ajustar marketing y revisar precios para "
            "evitar baja ocupación."
        )
    else:
        st.info("Demanda estimada: BAJA")
        recommendation = (
            "Recomendación: reducir riesgo operativo, optimizar presupuesto de marketing "
            "y considerar promociones."
        )

    st.write(recommendation)

    if probabilities:
        probability_df = pd.DataFrame(
            {
                "Clase": list(probabilities.keys()),
                "Probabilidad": list(probabilities.values()),
            }
        ).sort_values("Probabilidad", ascending=False)

        st.write("Probabilidades por clase")
        st.dataframe(probability_df, use_container_width=True)

        st.bar_chart(probability_df.set_index("Clase"))

    with st.expander("Ver variables enviadas al modelo"):
        st.dataframe(pd.DataFrame([result["input_features"]]), use_container_width=True)


# ==============================
# 7. Nota metodológica
# ==============================
st.divider()
st.caption(
    "Nota: Las variables históricas usadas por el modelo son simuladas para el MVP académico. "
    "El sistema no usa occupancy_pct ni tickets_sold como entrada para evitar fuga de información."
)


## 7. Probar que los archivos se crearon

Esta celda lista los archivos generados.


In [ ]:
!ls -lah model
!ls -lah app


## 8. Probar entrenamiento

Ejecuta esta celda solo si ya tienes `data/dataset.csv` en el repositorio.

Esto vuelve a entrenar el modelo y genera:

- `model/model.pkl`
- `model/metrics.txt`
- `model/metrics.json`


In [ ]:
# Ejecutar solo si ya existe data/dataset.csv
# !python model/train.py


## 9. Probar predicción

Ejecuta esta celda después de tener `model/model.pkl`.

Sirve para validar que `predict.py` funcione correctamente.


In [ ]:
# Ejecutar solo después de tener model/model.pkl
# !python model/predict.py


## 10. Ejecutar la app

En local o en un entorno compatible con Streamlit:

```bash
streamlit run app/app.py
```

En Colab, Streamlit requiere configuración adicional, por lo que lo más común es probar la app localmente o desplegarla después.


## 11. Commit recomendado

Después de ejecutar las celdas que escriben los archivos, sube los cambios así:

```bash
git add model/__init__.py
git add model/model.py
git add model/train.py
git add model/predict.py
git add app/app.py

git commit -m "Actualizar integración del modelo final en scripts y app"
git push origin main
```


## Validación final antes de subir a GitHub

Después de ejecutar el notebook, revisa que los archivos hayan quedado en las carpetas correctas:

```text
model/model.py
model/train.py
model/predict.py
model/__init__.py
app/app.py
```


In [ ]:
print("Archivos principales:")
!ls -lah model/model.py model/train.py model/predict.py model/__init__.py app/app.py
